# 01 — Corpus ingest: how raw rows become a build-ready corpus

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1) —
this notebook covers the **Documents** node at the head of the build-time flow,
everything that happens *before* text processing and embedding.

Nothing here opens a file path. Every read and write goes through the Kedro
**catalog**, which is what makes the same pipeline work against the 63-document
sample and a 200,000-row Hugging Face corpus without a code change.

```mermaid
flowchart TB
    CCN[("raw_ccnews_source\nHuggingFaceDataset · streaming · SHA-pinned\nor a local JSONL")]
    MIR[("raw_miracl_source\nHfFileDataset · JSONL.gz · SHA-pinned\nor a local JSONL")]
    CCN --> A1["snapshot_raw_ccnews"]
    A1 --> B1[("raw_ccnews\ndata/01_raw")]
    B1 --> C1["normalize_ccnews\nplain_text→text  requested_url→url"]
    C1 --> D1[("normalized_ccnews\ndata/02_intermediate")]
    MIR --> A2["snapshot_raw_miracl"]
    A2 --> B2[("raw_miracl\ndata/01_raw")]
    B2 --> C2["normalize_miracl\ntext→text  docid→url"]
    C2 --> D2[("normalized_miracl\ndata/02_intermediate")]
    D1 --> M["merge_documents\ndedup by id · ccn- precedes mir-"]
    D2 --> M
    M --> G[("merged_documents\ndata/02_intermediate")]
    G --> E["select_documents\nlanguage · metadata · cap"]
    E --> F[("documents\ndata/03_primary\nwhat a build consumes")]
    F -.-> H["index_build pipeline"]
```

**What you will see calculated below**

1. What normalisation actually removes, row by row, and why — shown for the
   CC-News branch (the rules are identical for MIRACL).
2. That document ids do not depend on row order — the property the whole
   reproducibility story rests on.
3. How `min_text_chars` trades corpus size against document quality.

## Bootstrap

`kedro jupyter lab` injects `catalog` automatically. This helper does the same thing
from *any* kernel — plain Jupyter, VS Code, or the headless executor that runs this
notebook in the test suite — so the file you are reading is the file CI verifies.

In [ ]:
from cybernaut_mini.notebook import kedro_catalog, project_root, run_pipeline

print("project root:", project_root())

catalog = kedro_catalog()
print("catalog entries:", sorted(catalog.keys()))

## Two deliberately messy sources

Real corpora are not clean. This demo writes two small datasets through the
catalog — one in CC-News column format, one in MIRACL corpus format — planting
failure modes the normaliser has to survive.

In the `prod` environment the same catalog entries are overridden by
`HuggingFaceDataset` (CC-News) and `HfFileDataset` (MIRACL corpus shards);
the nodes never change.

In [ ]:
GOOD = (
    "Analysts expect refinancing conditions to tighten materially over the coming "
    "quarter as corporate credit spreads widen."
)

# CC-News branch — column names: plain_text, requested_url, title, language, …
ccnews_rows = [
    {
        "plain_text": f"{GOOD} Case {i}.",
        "title": f"Market Outlook {i}",
        "requested_url": f"https://example.com/a-{i}",
        "language": "en",
        "published_date": "2024-01-01",
        "publisher": "example.com",
        "sitename": "Example",
        "author": "Jane Smith",
    }
    for i in range(4)
] + [
    # Too short — below min_text_chars, dropped.
    {"plain_text": "No comment.", "title": "", "requested_url": "https://example.com/short",
     "language": "en", "published_date": "2024-01-01",
     "publisher": "example.com", "sitename": "Example", "author": ""},
    # Duplicate url of a-0 — collapses to the first occurrence.
    {"plain_text": f"{GOOD} A duplicate of the first row.",
     "title": "Market Outlook 0", "requested_url": "https://example.com/a-0",
     "language": "en", "published_date": "2024-01-01",
     "publisher": "example.com", "sitename": "Example", "author": "Jane Smith"},
    # Messy whitespace — kept, but normalised.
    {"plain_text": "Ragged\n\n   spacing   and\ttabs throughout this otherwise fine passage body.",
     "title": "", "requested_url": "https://example.com/messy",
     "language": "en", "published_date": "2024-01-01",
     "publisher": "example.com", "sitename": "Example", "author": ""},
]

# MIRACL corpus branch — column names: docid, title, text
miracl_rows = [
    {"docid": "12#0", "title": "Photosynthesis",
     "text": "Photosynthesis is a process used by plants to convert light energy into chemical energy that can be stored and used."},
    {"docid": "42#1", "title": "Continental drift",
     "text": "Continental drift is the hypothesis that the Earth's continents have moved over geological time relative to each other."},
    # Too short — dropped by min_text_chars.
    {"docid": "99#0", "title": "", "text": "Too short."},
]

catalog.save("raw_ccnews_source", ccnews_rows)
catalog.save("raw_miracl_source", miracl_rows)
print(f"wrote {len(ccnews_rows)} CC-News rows and {len(miracl_rows)} MIRACL rows through the catalog")

## Run the acquisition pipeline

Six nodes, six catalog writes — two snapshots, two normalisation steps, one
merge, one select. Acquisition is a *separate* pipeline from `index_build` on
purpose: fetching is slow and rate-limited and should happen once, while an
index gets rebuilt many times over the same snapshot as settings are tuned.

In [ ]:
run_pipeline(
    "corpus_ingest",
    corpus_ccnews={
        "field_map": {
            "plain_text": "text",
            "requested_url": "url",
            "title": "title",
            "language": "language",
            "published_date": "published_at",
        },
        "metadata_fields": ["publisher", "sitename", "author"],
        "id_prefix": "ccn",
        "default_language": "en",
        "min_text_chars": 32,
    },
    corpus_miracl={
        "field_map": {"text": "text", "title": "title", "docid": "url"},
        "metadata_fields": ["docid"],
        "id_prefix": "mir",
        "default_language": "en",
        "min_text_chars": 32,
    },
    corpus_selection={"languages": None, "metadata_equals": None, "max_documents": None},
)
print("corpus_ingest complete")

## What each layer holds

Read all layers back **through the catalog**. Both branches are visible
independently, which is the payoff: you can inspect the CC-News normalisation
separately from the MIRACL normalisation, and see exactly what the merge added.

In [ ]:
raw_ccnews = catalog.load("raw_ccnews")
normalized_ccnews = catalog.load("normalized_ccnews")
raw_miracl = catalog.load("raw_miracl")
normalized_miracl = catalog.load("normalized_miracl")
merged = catalog.load("merged_documents")
primary = catalog.load("documents")

print(f"01_raw           raw_ccnews            {len(raw_ccnews):>3} rows   (verbatim CC-News source)")
print(f"02_intermediate  normalized_ccnews     {len(normalized_ccnews):>3} docs   (Document schema)")
print(f"01_raw           raw_miracl            {len(raw_miracl):>3} rows   (verbatim MIRACL source)")
print(f"02_intermediate  normalized_miracl     {len(normalized_miracl):>3} docs   (Document schema)")
print(f"02_intermediate  merged_documents      {len(merged):>3} docs   (CC-News + MIRACL, deduped)")
print(f"03_primary       documents             {len(primary):>3} docs   (build-ready)")
print()
print(f"CC-News: normalisation removed {len(raw_ccnews) - len(normalized_ccnews)} of {len(raw_ccnews)} rows")
print(f"MIRACL:  normalisation removed {len(raw_miracl) - len(normalized_miracl)} of {len(raw_miracl)} rows")

### Which CC-News rows were dropped, and why

The normaliser skips unusable rows rather than raising on them. That is a deliberate
choice: one malformed row out of 100,000 must not fail a multi-hour production build.
Here we reconstruct the reason for each drop in the CC-News branch.

In [ ]:
from cybernaut_mini.corpus import CorpusSourceConfig, make_document_id

ccnews_config = CorpusSourceConfig(
    field_map={"plain_text": "text", "requested_url": "url"},
    metadata_fields=["publisher", "sitename", "author"],
    id_prefix="ccn",
    min_text_chars=32,
)

kept_ids = {doc["id"] for doc in normalized_ccnews}
seen: set[str] = set()

print(f"{'verdict':<12} {'reason':<22} url")
print("-" * 74)
for row in raw_ccnews:
    text = " ".join(row["plain_text"].split())
    natural_key = row["requested_url"] or text
    doc_id = make_document_id(ccnews_config.id_prefix, natural_key)

    if len(text) < ccnews_config.min_text_chars:
        verdict, reason = "DROPPED", f"< {ccnews_config.min_text_chars} chars"
    elif doc_id in seen:
        verdict, reason = "DROPPED", "duplicate natural key"
    else:
        verdict, reason = "kept", ""
        seen.add(doc_id)

    print(f"{verdict:<12} {reason:<22} {row['requested_url'] or '(no url)'}")

print()
print(f"survivors: {len(kept_ids)}")

### Whitespace and derived titles

These corpora ship passages with no title, but the shard manifest, the shard summary,
and the `title\ntext` string that gets embedded all assume one exists. A title is
therefore derived from the first sentence, cut on a word boundary.

In [ ]:
for doc in normalized_ccnews:
    if "Ragged" in doc["text"]:
        print("raw text  :", repr(next(r["plain_text"] for r in raw_ccnews if "Ragged" in r["plain_text"])))
        print("normalised:", repr(doc["text"]))
        print()

print(f"{'id':<20} {'title':<62} chars")
print("-" * 92)
for doc in normalized_ccnews:
    print(f"{doc['id']:<20} {doc['title'][:60]:<62} {len(doc['title']):>3}")

## Dynamic 1 — ids do not depend on row order

This is the property everything else rests on. Ids are `blake2b` hashes of the row's
natural key (its `url`, or the body text when there is none), **not** sequential
counters. Re-fetching a corpus whose rows came back in a different order therefore
produces the same ids, the same shard assignments, and the same byte-for-byte index.

Sequential ids would renumber every document and silently invalidate every stored
relevance judgment.

In [ ]:
import random

from cybernaut_mini.corpus import normalize_rows

forward = [d.id for d in normalize_rows(ccnews_rows, ccnews_config)]

shuffled = list(ccnews_rows)
random.Random(0).shuffle(shuffled)
backward = [d.id for d in normalize_rows(shuffled, ccnews_config)]

print("original order :", forward[:3], "...")
print("shuffled order :", backward[:3], "...")
print()
print("identical:", forward == backward)
assert forward == backward, "ids must not depend on row order"

## Dynamic 2 — `min_text_chars` trades size against quality

The knob that decides how much of a public corpus survives. Sweeping it shows the
cost directly: too low and boilerplate and navigation text become "documents" that
pollute every shard's keyword profile; too high and you discard real content.

In [ ]:
print(f"{'min_text_chars':>15} {'documents kept':>16} {'mean text length':>18}")
print("-" * 52)
for threshold in (0, 16, 32, 80, 120, 400):
    swept = CorpusSourceConfig(
        field_map={"plain_text": "text", "requested_url": "url"},
        metadata_fields=["publisher", "sitename", "author"],
        id_prefix="ccn",
        min_text_chars=threshold,
    )
    docs = normalize_rows(ccnews_rows, swept)
    mean_len = sum(len(d.text) for d in docs) / len(docs) if docs else 0.0
    print(f"{threshold:>15} {len(docs):>16} {mean_len:>18.1f}")

print()
print("conf/prod uses 120: long enough to exclude nav text and stubs from a web corpus.")

## Dynamic 3 — selection narrows without re-fetching

`select_documents` runs *after* the merge, between the `merged_documents` and
`documents` layers. That ordering matters: narrowing the corpus never requires
downloading it again, so you can re-scope a build for free.

In [ ]:
from cybernaut_mini.pipelines.corpus_ingest.nodes import select_documents

print(f"{'selection':<44} {'documents':>10}")
print("-" * 56)
print(f"{'(no filter)':<44} {len(merged):>10}")

for label, params in [
    ("languages=['en']", {"languages": ["en"]}),
    ("max_documents=3", {"max_documents": 3}),
]:
    try:
        kept = select_documents(merged, params)
        print(f"{label:<44} {len(kept):>10}")
    except ValueError as exc:
        print(f"{label:<44} {'ERROR':>10}  {exc}")

## Takeaways

| Observation | Consequence |
|---|---|
| Ids are content-derived, not positional | A re-fetch in a different order still rebuilds the identical index |
| Unusable rows are skipped, not raised on | One bad row in 100,000 cannot fail a multi-hour build |
| Two sources with different column schemas | `field_map` in `CorpusSourceConfig` handles translation; the normaliser never branches |
| Merge deduplicates by id | A ccn- and mir- collision (impossible with disjoint prefixes) would silently keep the first occurrence |
| Selection sits after the merge | Re-scoping a corpus costs zero network calls |
| Every read and write went through `catalog` | Swapping a local source for a pinned Hugging Face repo is a `conf/prod` edit, not a code change |

Next: [`02_shard_anatomy.ipynb`](02_shard_anatomy.ipynb) takes the corpus this
produced and asks whether the shards built from it are actually coherent.